# Travel Agent Notebook

In [45]:
import os
from dotenv import load_dotenv

load_dotenv()

True

## Tools

In [ ]:
from langchain.tools import tool, ToolRuntime
from langchain.messages import ToolMessage
from langgraph.types import Command
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> str:
    """Search the web for information"""
    return tavily_client.search(query)

@tool
def update_trip_info(
    runtime: ToolRuntime,
    origin: str | None = None,
    destination: str | None = None,
    season: str | None = None,
    departure_date: str | None = None,
    return_date: str | None = None,
    budget: str | None = None
    ) -> Command:
    """Save any trip details the user has provided. Pass only the fields
    the user mentioned this turn and leave the rest as None."""

    updates = {k: v for k, v in {
        "origin": origin,
        "destination": destination,
        "season": season,
        "departure_date": departure_date,
        "return_date": return_date,
        "budget": budget,
        "messages": [ToolMessage("Success", tool_call_id=runtime.tool_call_id)]
    }.items() if v is not None}
    
    return Command(update=updates)

## Custom State

In [47]:
from langchain.agents import AgentState

class TripState(AgentState):
    origin: str
    destination: str
    season: str
    departure_date: str
    return_date: str
    budget: str

## System Prompt

In [48]:
system_prompt = """
You are an expert travel agent. Given the user's travel preferences and requirements,  
find the best potential travel destinations and activities as well as provide tips and 
helpful information when appropriate.

Use the web_search tool to find up to date information on potential destinations. 

When a user gives a trip detail (destination, season, dates, budget), call update_trip_info
to save it before replying.

"""

## Agent

In [49]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver  

agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search, update_trip_info],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)

In [50]:
from langchain.messages import HumanMessage

question = HumanMessage(content="Find a place to visit in Europe during December")

config = {"configurable": {"thread_id": "2"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="Find a place to visit in Europe during December")]},
    config
    )

In [51]:
print(response["messages"][-1].content)

Great idea—December in Europe offers a wonderful mix of festive markets, winter scenery, and (in the far north) magical aurora chances, plus some milder sun destinations if you prefer to escape the cold.

Here are good options across different vibes and budgets. I’ve noted quick what-to-expect and a couple of must-do ideas for each:

1) Vienna, Austria — classic Christmas market experience
- Why go: One of Europe’s most iconic Christmas vibes, with multiple markets (Rathausplatz, Schönbrunn, Freyung, Spittelberg) and plenty of seasonal treats.
- What to expect: Markets typically run from early to late December, with Schönbrunn starting in November and sometimes running into January; Rathausplatz market is the centerpiece and very photogenic.
- Must-dos: Glühwein, roasted chestnuts, carol concerts, ice skating at the Rathaus, a stroll along the festive city squares.
- Ideal timing: Mid-to-late December for an all-out market atmosphere.
- Quick tip: Book key sights early if you’re visiti

In [52]:
question = HumanMessage(content="Ok Prague sounds good. What language do they speak there?")

response = agent.invoke(
    {"messages": [question]},
    config,  
)

In [53]:
print(response["messages"][-1].content)

Great choice. In Prague (Czech Republic), the local language is Czech. However:

- English is widely spoken in tourist areas, hotels, restaurants, and at many attractions.
- Some people also speak German or Russian, but English is the most common for visitors.

Quick handy phrases (Czech):
- Hello/Good day: Dobrý den
- Please: Prosím
- Thank you: Děkuji
- Do you speak English?: Mluvíte anglicky?
- Yes/No: Ano / Ne
- Excuse me / Sorry: Promiňte

Would you like a short cheat sheet with pronunciation tips or a few key phrases tailored to your Prague itinerary?


In [54]:
snapshot = agent.get_state(config)
for m in snapshot.values["messages"]:
    print(m.type, m.content)

human Find a place to visit in Europe during December
ai 
tool Success
tool {"query": "Vienna Christmas markets December dates 2024 2025 Christmas market Vienna", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.intothebloom.com/vienna-christmas-markets", "title": "Vienna Christmas Markets Guide 2025: Dates, Tips + Map - Into the Bloom", "content": "⏰ 2025 Christmas market opening times:  \nMonday – Friday: 11 am – 9 pm  \nSaturday – Sunday: 10 am – 9 pm  \n24th December: 11 am – 4 pm  \n25th- 30th December: 11 am – 8 pm  \n31st December: 11 am – 6 pm\n\nThe Belvedere Christmas market is much smaller than the one at Schönbrunn Palace, but there’s something about a market with a palace as its backdrop that makes it feel magical, no matter its size. [...] 📅 2025 Christmas market opening dates:   \n21st November – 23rd December 2025\n\n⏰ 2025 Christmas market opening times:  \nDaily from 12 pm to 8 pm\n\nAbout a 20-minute walk from Belvedere Pala